# Yearly raster time series, Mann-Kendall Tau, and STANDARD

The calculation flow is: create an ordered in-memory raster group from separate annual files, prepare the yearly time series and per-year NaN masks, calculate Mann-Kendall Tau, then apply STANDARD to the Tau image. A histogram review of the Tau and standardized-Tau distributions is the final step. The notebook does not classify results; it writes the standardized Tau raster that the downstream classification step consumes.

In [1]:
### Import Packages ###
import os, sys
from pathlib import Path
import math, multiprocessing, os, re
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.windows import Window

In [2]:
### Filename helper ###

def get_year(filename):
    """Extract a four-digit year from names such as ``1990_area_1km.tif``."""
    filename = Path(filename).name
    match = re.search(r"(?<!\d)((?:19|20)\d{2})(?!\d)", filename)
    if not match:
        raise ValueError(
            f"Could not extract a four-digit year from {filename!r}. "
            "Expected a name such as '1990_area_1km.tif'."
        )
    return int(match.group(1))


def get_ordered_input_files(input_dir, input_pattern):
    """Find yearly rasters and sort them by the four-digit year in each name."""
    input_paths = list(Path(input_dir).glob(input_pattern))
    if not input_paths:
        raise FileNotFoundError(f"No rasters match {Path(input_dir) / input_pattern}")
    keyed_paths = [(get_year(path.name), path) for path in input_paths]
    keyed_paths.sort(key=lambda item: (item[0], item[1].name.lower()))
    years = [year for year, _ in keyed_paths]
    if len(years) != len(set(years)):
        raise ValueError("More than one input raster matched for a year; narrow input_pattern to one raster per year")
    return [path for _, path in keyed_paths]



def build_workflow_paths(main_workspace, metric_type):
    """Build the fixed input and output paths from the workspace and metric."""
    workspace = Path(main_workspace)
    input_dir = workspace / "3_MovingWindow" / f"mw_{metric_type}"
    input_pattern = f"*_{metric_type}_1km.tif"
    analysis_dir = workspace / "4_TS_and_TI"
    return {
        "input_dir": input_dir,
        "input_pattern": input_pattern,
        "tau_path": analysis_dir / "TS" / metric_type / f"{metric_type}_raster_group_MK_tau.tif",
        "mask_nan_dir": analysis_dir / "TS_mask" / f"{metric_type}_mask",
        "standardized_tau_path": analysis_dir / "TS_zscore" / metric_type / f"{metric_type}_MK_tau_z.tif",
    }



def validate_parameters(main_workspace, metric_type, min_observations,
                        multiprocessing_workers, tile_rows, tile_cols):
    """Check user-selected settings before starting raster processing."""
    # The workspace root must already exist. Do not create it: a missing
    # main_workspace is a setup error, and auto-creating it would silently
    # make empty output trees at an unintended location (e.g. a stray
    # Windows-style path inside a Linux container).
    if not Path(main_workspace).is_dir():
        raise FileNotFoundError(
            f"main_workspace does not exist or is not a directory: {main_workspace}. "
            "Create or mount it before running; it is not created automatically."
        )
    if metric_type not in {"area", "edge", "pn"}:
        raise ValueError("metric_type must be 'area', 'edge', or 'pn'")
    if min_observations < 2:
        raise ValueError("min_observations must be at least 2")
    if multiprocessing_workers < 1 or tile_rows < 1 or tile_cols < 1:
        raise ValueError("multiprocessing_workers, tile_rows, and tile_cols must be positive")



def validate_zero_nodata_semantics(sources):
    """Reject ambiguous inputs that declare zero as NoData without a mask.

    A separate per-dataset or alpha mask can distinguish valid zeros from
    missing cells. If zero is the only NoData signal, those meanings cannot
    be recovered reliably from pixel values alone.
    """
    for source in sources:
        flags = {flag.name for flag in source.mask_flag_enums[0]}
        has_explicit_mask = bool({"per_dataset", "alpha"} & flags)
        if source.nodata == 0 and "nodata" in flags and not has_explicit_mask:
            raise ValueError(
                f"{source.name} declares 0 as NoData without a separate validity mask. "
                "Valid zeros and missing cells would be indistinguishable. "
                "Encode missing cells as NaN or a nonzero NoData value, or provide "
                "an explicit raster validity mask before running this workflow."
            )


### Parameters

Edit the project workspace, metric, and processing settings here. Input and output folders follow the project's fixed folder structure and are built by a helper function below.

In [3]:
# Check Main Directory and list files in Data folder
import glob
print("Current directory:", os.getcwd())
files = glob.glob("/home/gisuser/code/data/*")
print(files)

Current directory: /home/gisuser/code
['/home/gisuser/code/data/sites_selection_data', '/home/gisuser/code/data/3_MovingWindow', '/home/gisuser/code/data/4_TS_and_TI']


In [4]:
### Parameters ###

# Project and metric
main_workspace = "./data"  # Path(r"D:\GitHub\typology\data")
metric_type = "edge"  # Choose: "area", "edge", or "pn"

# Mann-Kendall settings
min_observations = 10  # Editable Python cutoff: minimum valid years required per pixel

# Multiprocessing settings: use 80% of available logical CPUs when enabled
loc_multiprocessing = "./code/ipynb_workflow/mk_calc_worker.py"  # mk_calc_worker.py file location 
use_multiprocessing = True
available_cores = os.cpu_count() or 1
multiprocessing_workers = max(1, math.floor(available_cores * 0.8)) if use_multiprocessing else 1
tile_rows = 128
tile_cols = 256

## Workflow stages and TerrSet-aligned choices

1. **Raster group and yearly series:** read the separate annual rasters in year order (`1990_area`, etc.). These are the observations for a metric; they are not separate MK outputs. Grid alignment is checked, and a per-year `mask_nan` file records missing cells (`1`) versus data (`0`). No stack or group manifest is written.
2. **Mann-Kendall Tau:** calculate one Tau image for the selected metric. The code uses `S / [n(n-1)/2]` over valid observation pairs; equal-valued pairs add zero to S. This is the Tau-only output. Significance Z/p, CMK, and Theil-Sen are not calculated.
3. **STANDARD(Tau):** apply `(Tau - valid-cell mean) / valid-cell standard deviation` to valid Tau cells. The divisor is `N` as selected here; the supplied TerrSet description does not specify `N` versus `N-1`.
4. **Histogram review:** inspect the raw Tau and standardized-Tau distributions, and view the chosen +/- threshold lines. This is a final review step; the notebook does not apply the threshold or recode the central range.

The supplied KENDALL notes require at least 10 observations for an appropriate result. The Python setting applies this minimum per pixel; the notes do not specify TerrSet's exact per-pixel missing-observation behavior. Inputs must distinguish valid zero from NoData. Missing values are excluded; a valid Tau of zero remains data. No classification is produced.

In [5]:
### Build fixed input and output paths ###

validate_parameters(
    main_workspace, metric_type, min_observations,
    multiprocessing_workers, tile_rows, tile_cols
)
workflow_paths = build_workflow_paths(main_workspace, metric_type)
input_files = get_ordered_input_files(
    workflow_paths["input_dir"], workflow_paths["input_pattern"]
)
print(f"Input folder: {workflow_paths['input_dir']}")
print(f"Matched yearly rasters: {len(input_files)}")
print(f"Years: {get_year(input_files[0].name)} - {get_year(input_files[-1].name)}")
print(f"Tau output: {workflow_paths['tau_path']}")
print(f"NaN masks: {workflow_paths['mask_nan_dir']}")
print(f"Standardized Tau: {workflow_paths['standardized_tau_path']}")
print(f"Metric: {metric_type}")
print(f"Multiprocessing: {use_multiprocessing}; workers: {multiprocessing_workers}; "
      f"tile size: {tile_rows} rows x {tile_cols} columns")

Input folder: data/3_MovingWindow/mw_edge
Matched yearly rasters: 31
Years: 1990 - 2020
Tau output: data/4_TS_and_TI/TS/edge/edge_raster_group_MK_tau.tif
NaN masks: data/4_TS_and_TI/TS_mask/edge_mask
Standardized Tau: data/4_TS_and_TI/TS_zscore/edge/edge_MK_tau_z.tif
Metric: edge
Multiprocessing: True; workers: 19; tile size: 128 rows x 256 columns


In [6]:
### Mann-Kendall worker import ###

# The per-tile Tau kernel lives in mk_calc_worker.py next to this notebook.
# Keeping it in a real importable module (rather than an embedded source
# string) lets the 'spawn' processes used on Windows/Docker import it directly.

worker_dir = os.path.dirname(os.path.abspath(loc_multiprocessing))
if worker_dir not in sys.path:
    sys.path.insert(0, worker_dir)

from mk_calc_worker import calculate_mk_for_tile

In [9]:
### Stage 1: create the raster group

def create_raster_group(input_paths, metric_type):
    """Create an ordered in-memory group from separate annual raster files.

    The returned object contains the paths, years, and band-style labels such
    as ``1990_area``. No manifest or raster data copy is written.
    """
    keyed_paths = [(get_year(Path(path).name), Path(path).resolve()) for path in input_paths]
    keyed_paths.sort(key=lambda item: (item[0], item[1].name.lower()))
    ordered_paths = [path for _, path in keyed_paths]
    years = [year for year, _ in keyed_paths]
    if not ordered_paths:
        raise ValueError("At least one annual input raster is required")
    if len(years) != len(set(years)):
        raise ValueError("More than one raster was provided for a year")
    missing_paths = [path for path in ordered_paths if not path.is_file()]
    if missing_paths:
        raise FileNotFoundError(f"Raster group contains missing inputs: {missing_paths}")
    labels = [f"{year}_{metric_type}" for year in years]
    print(f"In-memory raster group contains {len(ordered_paths)} annual rasters")
    return {"input_paths": ordered_paths, "years": years,
            "time_series_labels": labels, "metric_type": metric_type}


def run_raster_group_stage():
    """Create the ordered annual raster group from the notebook parameters."""
    return create_raster_group(input_files, metric_type)

In [10]:
### Run Stage 1: create the raster group

raster_group_results = run_raster_group_stage()
print(f"Raster group contains {len(raster_group_results['input_paths'])} annual inputs")

In-memory raster group contains 31 annual rasters
Raster group contains 31 annual inputs


In [11]:
### Stage 2: prepare the yearly time series and NaN masks

def create_time_series(raster_group_results, mask_nan_dir, tile_rows, tile_cols):
    """Validate annual rasters and write a separate missing-data mask per year.

    Annual rasters stay separate and form the time series by their ordered
    paths. Windowed reads avoid building or saving a multiband stack. In each
    mask, 1 means missing/NaN and 0 means data is present, including valid zero.
    """
    input_paths = raster_group_results["input_paths"]
    years = raster_group_results["years"]
    if not input_paths:
        raise ValueError("The raster group contains no annual rasters")
    sources = [rasterio.open(path) for path in input_paths]
    mask_destinations = []
    try:
        validate_zero_nodata_semantics(sources)
        reference = sources[0]
        for path, source in zip(input_paths, sources):
            same_grid = ((source.width, source.height, source.transform, source.crs) ==
                         (reference.width, reference.height, reference.transform, reference.crs))
            if not same_grid:
                raise ValueError(f"Input raster grid differs from the first raster: {path}")

        mask_nan_dir = Path(mask_nan_dir)
        mask_nan_dir.mkdir(parents=True, exist_ok=True)
        mask_nan_paths = [mask_nan_dir / f"{raster_group_results['metric_type']}_{year}_mask_nan.tif"
                          for year in years]
        mask_profile = reference.profile.copy()
        # Striped (non-tiled) output so each per-year mask matches the
        # striped input layout; clear any inherited tiling keys.
        mask_profile.update(driver="GTiff", count=1, dtype="uint8", nodata=None,
                            compress="deflate", tiled=False)
        mask_profile.pop("blockxsize", None)
        mask_profile.pop("blockysize", None)
        mask_destinations = [rasterio.open(path, "w", **mask_profile) for path in mask_nan_paths]
        # Process full-width row bands. A compressed striped GeoTIFF is
        # written one strip at a time, so each write spans the whole image
        # width; column tiling would force read-modify-write of strips.
        row_blocks = math.ceil(reference.height / tile_rows)
        progress_interval = max(1, row_blocks // 20)
        for block_index, row in enumerate(range(0, reference.height, tile_rows), start=1):
            height = min(tile_rows, reference.height - row)
            window = Window(0, row, reference.width, height)
            for source, destination in zip(sources, mask_destinations):
                values = source.read(1, window=window)
                valid = (source.read_masks(1, window=window) > 0) & np.isfinite(values)
                destination.write((~valid).astype(np.uint8), 1, window=window)
            if block_index % progress_interval == 0 or block_index == row_blocks:
                print(f"NaN-mask rows written: {min(row + tile_rows, reference.height):,}/{reference.height:,}")

        print(f"Per-year mask_nan rasters saved: {mask_nan_dir}")
        return {"input_paths": input_paths, "years": years,
                "time_series_labels": raster_group_results["time_series_labels"],
                "mask_nan_paths": mask_nan_paths, "profile": reference.profile.copy(),
                "metric_type": raster_group_results["metric_type"]}
    finally:
        for destination in mask_destinations:
            destination.close()
        for source in sources:
            source.close()


def run_time_series_stage(raster_group_results):
    """Prepare the yearly raster time series and create its NaN masks."""
    return create_time_series(
        raster_group_results=raster_group_results,
        mask_nan_dir=workflow_paths["mask_nan_dir"],
        tile_rows=tile_rows,
        tile_cols=tile_cols,
    )

In [12]:
import os, resource
print("tile_rows =", tile_rows)                 # did it get bumped up?
print("n_years   =", len(input_files))
import rasterio
with rasterio.open(input_files[0]) as s:
    print("width x height:", s.width, "x", s.height)
    print("per-strip MB  :", tile_rows * s.width * len(input_files) * 4 / 1e6)
print("GDAL_CACHEMAX:", os.environ.get("GDAL_CACHEMAX"))

tile_rows = 128
n_years   = 31
width x height: 74667 x 96605
per-strip MB  : 1185.114624
GDAL_CACHEMAX: None


In [ ]:
### Run Stage 2: prepare the time series
#Open each annual raster.
#Validate they share the same grid.
#For each year, write a NaN mask (a uint8 file where 1 = missing, 0 = data present).


time_series_results = run_time_series_stage(raster_group_results)
print(f"Prepared {len(time_series_results['time_series_labels'])} yearly inputs")
print(f"Created {len(time_series_results['mask_nan_paths'])} per-year NaN masks")

NaN-mask rows written: 4,736/96,605
NaN-mask rows written: 9,472/96,605
NaN-mask rows written: 14,208/96,605
NaN-mask rows written: 18,944/96,605
NaN-mask rows written: 23,680/96,605
NaN-mask rows written: 28,416/96,605
NaN-mask rows written: 33,152/96,605
NaN-mask rows written: 37,888/96,605
NaN-mask rows written: 42,624/96,605
NaN-mask rows written: 47,360/96,605
NaN-mask rows written: 52,096/96,605


: 

: 

: 

: 

...

: 

: 

: 

: 

: 

In [ ]:
### Stage 3: Mann-Kendall Tau

def merge_image_statistics(count_a, mean_a, m2_a, values):
    """Merge finite Tau values into whole-image mean and variance statistics."""
    valid_values = values[np.isfinite(values)].astype(np.float64, copy=False)
    count_b = valid_values.size
    if count_b == 0:
        return count_a, mean_a, m2_a
    mean_b = float(valid_values.mean())
    m2_b = float(np.square(valid_values - mean_b).sum())
    if count_a == 0:
        return count_b, mean_b, m2_b
    delta = mean_b - mean_a
    count = count_a + count_b
    mean = mean_a + delta * count_b / count
    m2 = m2_a + m2_b + delta * delta * count_a * count_b / count
    return count, mean, m2


def calculate_mann_kendall_tau(time_series_results, tau_path, min_observations,
                               use_multiprocessing, multiprocessing_workers,
                               tile_rows, tile_cols):
    """Calculate Tau from separate annual rasters and return image statistics.

    Each worker reads its own spatial window from every annual input, so only
    tile coordinates and window offsets cross the process boundary. Pixels with
    fewer than ``min_observations`` valid years are written as NoData.
    Whole-image Tau population standard deviation is accumulated for STANDARD.
    """
    tile_worker = calculate_mk_for_tile
    input_paths = tuple(str(Path(path)) for path in time_series_results["input_paths"])
    sources = [rasterio.open(path) for path in input_paths]
    tau_path = Path(tau_path)
    image_count = 0
    tau_mean = 0.0
    tau_m2 = 0.0
    pool = None
    try:
        validate_zero_nodata_semantics(sources)
        reference = sources[0]
        profile = reference.profile.copy()
        nodata_value = -9999.0  # Exported NoData marker; Tau values are limited to [-1, 1]
        # Write a striped (non-tiled) single-band GeoTIFF so the output
        # layout matches the striped inputs. tiled is explicitly False and
        # any inherited blockxsize/tiling keys are cleared.
        profile.update(count=1, dtype="float32", nodata=nodata_value,
                       compress="deflate", tiled=False)
        profile.pop("blockxsize", None)
        profile.pop("blockysize", None)
        image_height, image_width = reference.height, reference.width

        # Choose a read pattern from the reference raster's internal layout.
        # A striped GeoTIFF stores whole rows, so its block width equals the
        # image width. Reading a narrow column window from a strip still pulls
        # the full scanline from disk, so column tiling only adds redundant
        # reads. For striped inputs, read full-width row bands and tile by rows
        # only; for internally tiled inputs, align both dimensions to blocks.
        block_shape = reference.block_shapes[0]
        native_rows, native_cols = int(block_shape[0]), int(block_shape[1])
        is_striped = native_cols >= image_width
        if is_striped:
            step_rows = max(native_rows, (tile_rows // native_rows) * native_rows or native_rows)
            step_cols = image_width  # full-width bands; no column tiling for strips
            print(f"Striped input (block {native_rows} x {native_cols}); "
                  f"reading full-width row bands of {step_rows} rows")
        else:
            step_rows = max(native_rows, (tile_rows // native_rows) * native_rows or native_rows)
            step_cols = max(native_cols, (tile_cols // native_cols) * native_cols or native_cols)
            print(f"Tiled input (block {native_rows} x {native_cols}); "
                  f"tile step: {step_rows} rows x {step_cols} columns")

        def tile_data():
            """Yield lightweight tile descriptors for worker-side reads."""
            for row in range(0, image_height, step_rows):
                block_height = min(step_rows, image_height - row)
                for col in range(0, image_width, step_cols):
                    block_width = min(step_cols, image_width - col)
                    window_offsets = (col, row, block_width, block_height)
                    yield row, col, window_offsets, input_paths, min_observations

        block_count = math.ceil(image_height / step_rows) * math.ceil(image_width / step_cols)
        worker_count = min(multiprocessing_workers, block_count)
        if use_multiprocessing and worker_count > 1:
            # Hand several tiles to each worker per dispatch to cut per-task
            # IPC and scheduling overhead when there are many small tiles.
            chunksize = max(1, block_count // (worker_count * 8))
            pool = multiprocessing.get_context("spawn").Pool(processes=worker_count)
            tile_results = pool.imap_unordered(tile_worker, tile_data(), chunksize=chunksize)
            print(f"Mann-Kendall processing with {worker_count} workers; chunksize {chunksize}")
        else:
            tile_results = map(tile_worker, tile_data())
            print("Mann-Kendall processing sequentially")

        completed_blocks = 0
        tau_path.parent.mkdir(parents=True, exist_ok=True)
        with rasterio.open(tau_path, "w", **profile) as tau_destination:
            for row, col, tau in tile_results:
                block_height, block_width = tau.shape
                window = Window(col, row, block_width, block_height)
                tau_destination.write(
                    np.where(np.isfinite(tau), tau, nodata_value).astype(np.float32),
                    1, window=window
                )
                image_count, tau_mean, tau_m2 = merge_image_statistics(
                    image_count, tau_mean, tau_m2, tau
                )
                completed_blocks += 1
                if completed_blocks % max(1, block_count // 20) == 0 or completed_blocks == block_count:
                    print(f"Mann-Kendall tiles: {completed_blocks:,}/{block_count:,}")

        if pool is not None:
            pool.close()
            pool.join()
            pool = None
        if image_count == 0:
            raise ValueError("No pixels met the minimum valid-year requirement")
        tau_std = math.sqrt(tau_m2 / image_count)
        return {"tau_path": tau_path, "tau_mean": tau_mean, "tau_std": tau_std,
                "valid_pixels": image_count, "profile": profile, "nodata": nodata_value}
    finally:
        if pool is not None:
            pool.terminate()
            pool.join()
        for source in sources:
            source.close()


def run_mann_kendall_tau_stage(time_series_results):
    """Run Tau analysis on the yearly raster inputs by spatial windows."""
    return calculate_mann_kendall_tau(
        time_series_results=time_series_results,
        tau_path=workflow_paths["tau_path"],
        min_observations=min_observations,
        use_multiprocessing=use_multiprocessing,
        multiprocessing_workers=multiprocessing_workers,
        tile_rows=tile_rows,
        tile_cols=tile_cols,
    )

In [ ]:
### Run Stage 3: calculate Mann-Kendall Tau

mk_results = run_mann_kendall_tau_stage(time_series_results)
print(f"Mann-Kendall Tau raster: {mk_results['tau_path']}")
print(f"Whole-image Tau mean: {mk_results['tau_mean']:.6g}")
print(f"Whole-image Tau standard deviation: {mk_results['tau_std']:.6g}")
print(f"Valid Tau pixels: {mk_results['valid_pixels']:,}")

In [ ]:
### Stage 4: STANDARD(Tau)

def standardize_tau(mk_results, output_path, tile_rows, tile_cols):
    """Apply the TerrSet STANDARD formula to the Mann-Kendall Tau raster."""
    tau_mean = mk_results["tau_mean"]
    tau_std = mk_results["tau_std"]
    if tau_std == 0:
        raise ValueError("Tau standard deviation is zero; STANDARD cannot rescale a constant Tau image")
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    profile = mk_results["profile"].copy()
    nodata_value = mk_results["nodata"]
    image_height, image_width = profile["height"], profile["width"]
    # Full-width row bands: the Tau raster is striped, so read and write one
    # strip-aligned band at a time instead of column sub-windows.
    with rasterio.open(mk_results["tau_path"]) as tau_source, \
         rasterio.open(output_path, "w", **profile) as standardized_destination:
        for row in range(0, image_height, tile_rows):
            block_height = min(tile_rows, image_height - row)
            window = Window(0, row, image_width, block_height)
            tau = tau_source.read(1, window=window, masked=True).astype(np.float32).filled(np.nan)
            valid = np.isfinite(tau)
            standardized = np.full(tau.shape, nodata_value, dtype=np.float32)
            standardized[valid] = ((tau[valid] - tau_mean) / tau_std).astype(np.float32)
            standardized_destination.write(standardized, 1, window=window)
    print(f"STANDARD(Tau) saved: {output_path}")
    return Path(output_path)


In [ ]:
### Run Stage 4: standardize Tau

standardized_tau_path = standardize_tau(
    mk_results, workflow_paths["standardized_tau_path"], tile_rows, tile_cols
)

## Stage 5: typology classification threshold and histogram review

The `zscore_threshold` below is the standardized-Tau cutoff for the downstream typology classification (applied in a later step, not here). This notebook only uses it to draw the `+/-` reference lines on the histogram so you can see where the classification cut will fall against the distribution. Adjust the threshold, re-run the histogram, and inspect the lines. The histogram does not apply the threshold or modify any raster.

In [ ]:
### Typology classification threshold (used by the downstream classification step) ###

# Standardized-Tau cutoff. The histogram below only displays this as +/- lines;
# the actual threshold-based classification is applied in a later step.
zscore_threshold = 0.5
if zscore_threshold < 0:
    raise ValueError("zscore_threshold must be non-negative")

In [ ]:
### Stage 5: histogram review of the standardized-Tau distribution

import matplotlib.pyplot as plt


def sample_raster_values(raster_path, max_samples=1_000_000):
    """Read a bounded regular sample from a raster for plotting."""
    with rasterio.open(raster_path) as source:
        factor = max(1, int(math.ceil(math.sqrt((source.width * source.height) / max_samples))))
        sampled = source.read(1,
            out_shape=(max(1, source.height // factor), max(1, source.width // factor)),
            resampling=Resampling.nearest, masked=True)
        return sampled.compressed()


def plot_tau_histogram(mk_results, zscore_threshold, bins=50, max_samples=1_000_000):
    """Plot raw Tau and standardized Tau; print whole-image and sampled statistics."""
    raw_sample = sample_raster_values(mk_results["tau_path"], max_samples)
    if raw_sample.size == 0:
        raise ValueError("No valid Tau pixels were found for the histogram")
    tau_std = mk_results["tau_std"]
    standardized_sample = ((raw_sample - mk_results["tau_mean"]) / tau_std
                           if tau_std else np.zeros_like(raw_sample))
    print(f"Whole-image valid pixel count: {mk_results['valid_pixels']:,}")
    print(f"Whole-image Tau mean: {mk_results['tau_mean']:.6g}")
    print(f"Whole-image Tau standard deviation: {tau_std:.6g}")
    print(f"Histogram sample count: {raw_sample.size:,}")
    print(f"Selected standardized-Tau threshold: +/-{zscore_threshold}")
    print("Sampled Tau min / median / max:", np.min(raw_sample), np.median(raw_sample), np.max(raw_sample))
    print("Sampled Tau percentiles (5, 25, 50, 75, 95):", np.percentile(raw_sample, [5, 25, 50, 75, 95]))
    histogram_counts, histogram_edges = np.histogram(raw_sample, bins=bins)
    print("Histogram bin edges:", histogram_edges)
    print("Histogram counts:", histogram_counts)
    figure, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(raw_sample, bins=bins, color="steelblue", edgecolor="white")
    axes[0].axvline(mk_results["tau_mean"], color="darkred", linestyle="--",
                    label=f"whole-image mean = {mk_results['tau_mean']:.3f}")
    axes[0].set(title="Raw Mann-Kendall Tau", xlabel="Tau", ylabel="Sampled pixel count")
    axes[0].legend()
    axes[1].hist(standardized_sample, bins=bins, color="seagreen", edgecolor="white")
    axes[1].axvline(0, color="black", linestyle=":", label="image mean")
    axes[1].axvline(zscore_threshold, color="darkred", linestyle="--",
                    label=f"threshold = +/-{zscore_threshold:g}")
    axes[1].axvline(-zscore_threshold, color="darkred", linestyle="--")
    axes[1].set(title="STANDARD(Tau)", xlabel="Standardized Tau", ylabel="Sampled pixel count")
    axes[1].legend()
    plt.tight_layout()
    plt.show()
    return {"raw_sample": raw_sample, "standardized_sample": standardized_sample}


In [ ]:
### Run Stage 5: review the standardized-Tau histogram and threshold reference

histogram_results = plot_tau_histogram(mk_results, zscore_threshold)

## Output files

All outputs are under `D:/GitHub/typology/data/4_TS_and_TI/`:

- `TS/<metric>/`: `<metric>_raster_group_MK_tau.tif`. The input raster group is kept in memory; no group manifest or multiband time-series raster is saved.
- `TS_mask/<metric>_mask/`: one `<metric>_<year>_mask_nan.tif` per year. Each file is `1` where that year's input is NoData/NaN, and `0` where data exists; valid pixel values of zero remain valid. A separate mask is necessary because valid coverage can change annually; valid zero values remain data.
- `TS_zscore/<metric>/`: `<metric>_MK_tau_z.tif`, standardized from valid Tau cells using the image mean and population standard deviation (`N`). The TerrSet STANDARD description supplied specifies mean subtraction and division by standard deviation but does not state the divisor; `N` is the chosen interpretation. The editable `zscore_threshold` parameter is used only by the final histogram review, where it is drawn as `+/-` reference lines on the standardized-Tau distribution. The histogram only displays the threshold; it does not apply it, set central values to zero, or change Tau or STANDARD. The threshold-based classification itself happens in the downstream step, not in this notebook. The notebook writes one Tau raster and one standardized raster per selected metric; run it for area, edge, and pn to create those three outputs.

Pixels with fewer than 10 valid annual observations are NoData in the Tau output (stored as -9999 with NoData metadata). Valid Tau values of zero remain zero. The same NoData marker is preserved in the standardized-Tau output; internal calculations use NaN for missing values. The provided TerrSet note says at least 10 observations are required but does not specify whether TerrSet applies this rule pixel by pixel. The multiprocessing helper is created in the system temporary folder. Statistical significance Z/p and classification are not produced.